In [ ]:
!pip install pyspark -q

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, count, desc, current_timestamp, regexp_replace
from pyspark.sql.types import DoubleType

spark = SparkSession.builder \
    .appName("Lab_DataMesh_CEAPS") \
    .config("spark.sql.repl.eagerEval.enabled", True) \
    .getOrCreate()

print("Infraestrutura de dados instanciada com sucesso.")


In [ ]:
df_monolito = spark.read.csv("CEAPS_2008_2019.csv", header=True, sep=",", inferSchema=True)

print("Esquema do Monolito Original:")
df_monolito.printSchema()


In [ ]:
from pyspark.sql.functions import col

dominio_politico_raw = df_monolito.select(
    col("SENADOR")
).dropDuplicates(["SENADOR"])

dominio_financeiro_raw = df_monolito.select(
    col("SENADOR"),
    col("ANO"),
    col("MES"),
    col("TIPO_DESPESA"),
    col("CNPJ_CPF"),
    col("FORNECEDOR"),
    col("VALOR_REEMBOLSADO")
)

In [ ]:
from pyspark.sql.types import DoubleType
from pyspark.sql.functions import regexp_replace, col, sum, count, current_timestamp

df_financeiro_limpo = dominio_financeiro_raw \
    .filter(col("VALOR_REEMBOLSADO").rlike(r"^[0-9\.,\-]+$")) \
    .withColumn("VALOR_REEMBOLSADO_CLEAN", regexp_replace(col("VALOR_REEMBOLSADO"), ",", ".").cast(DoubleType())) \
    .filter(col("VALOR_REEMBOLSADO_CLEAN").isNotNull())

dp_despesas_agregadas = df_financeiro_limpo \
    .groupBy("SENADOR", "ANO", "TIPO_DESPESA") \
    .agg(
        sum("VALOR_REEMBOLSADO_CLEAN").alias("TOTAL_REEMBOLSADO"),
        count("CNPJ_CPF").alias("QTD_TRANSACOES")
    ) \
    .withColumn("DP_DATA_ATUALIZACAO", current_timestamp())

print("Data Product Financeiro Disponibilizado para a Malha:")
dp_despesas_agregadas.show(5, truncate=False)


In [ ]:
from pyspark.sql.functions import desc

df_malha_transparencia = dp_despesas_agregadas.join(
    dominio_politico_raw,
    on="SENADOR",
    how="inner"
)

resultado_final = df_malha_transparencia \
    .groupBy("SENADOR", "TIPO_DESPESA") \
    .agg(sum("TOTAL_REEMBOLSADO").alias("CUSTO_TOTAL_SENADOR")) \
    .orderBy(desc("CUSTO_TOTAL_SENADOR"))

print("Top 10 Maiores Custos - Cruzamento Inter-Domínios:")
resultado_final.show(10, truncate=False)


In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lag, round, col

df_tendencia = dp_despesas_agregadas.groupBy("ANO") \
    .agg(sum("TOTAL_REEMBOLSADO").alias("CUSTO_ANUAL"))

windowSpec = Window.orderBy("ANO")

df_crescimento = df_tendencia \
    .withColumn("CUSTO_ANO_ANTERIOR", lag("CUSTO_ANUAL", 1).over(windowSpec)) \
    .withColumn("CRESCIMENTO_PERCENTUAL",
                round(((col("CUSTO_ANUAL") - col("CUSTO_ANO_ANTERIOR")) / col("CUSTO_ANO_ANTERIOR")) * 100, 2))

df_crescimento.show()


In [ ]:
df_monolito.select("TIPO_DESPESA").distinct().show(truncate=False)

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, sum, round

window_senador = Window.partitionBy("SENADOR")

df_proporcao = resultado_final \
    .withColumn("TOTAL_GERAL_SENADOR", sum("CUSTO_TOTAL_SENADOR").over(window_senador)) \
    .withColumn("PERCENTUAL_DO_TOTAL",
                round((col("CUSTO_TOTAL_SENADOR") / col("TOTAL_GERAL_SENADOR")) * 100, 2)) \
    .orderBy("SENADOR", col("PERCENTUAL_DO_TOTAL").desc())

print("Proporção de Gastos por Tipo de Despesa para cada Senador:")
df_proporcao.select("SENADOR", "TIPO_DESPESA", "CUSTO_TOTAL_SENADOR", "PERCENTUAL_DO_TOTAL").show(15, truncate=False)